# Single-cell RNA-seq analysis, step by step

**SupBioTech M2 practical** · human skin wound healing · scRNA-seq only

In this practical we analyse real single-cell RNA-seq data from the study *"Spatiotemporal single-cell roadmap of human skin wound healing"* (GEO accession **GSE241132**). We follow the whole life of a scRNA-seq dataset, from the raw count matrix to annotated cell types, and we **slow down at every step** to understand *why* we do it, *what* the numbers mean and *what happens if we do it differently*.

The goal is not to run a pipeline. The goal is that, at the end, you can explain every line of it.

## How to use this notebook

Every section follows the same rhythm, announced by a small label at the beginning of the block:

> **Concept** – the idea, explained in plain words. Read it before running any code.

> **Run** – a code cell you simply execute and then *look at the output*. Ask yourself: what is this telling me?

> **Break it** – an experiment where we change a parameter on purpose, to see what changes (and what does not).

> **Question** – a question to answer *before* opening the hidden answer.

> **Take-home** – the one-sentence summary to remember.

The answers to the questions are hidden inside a small arrow (**"Show answer"**). Click only after you have tried to answer, even if you are not sure.

### Mini survival kit for Jupyter and Python

- Run a cell with **Shift + Enter**. Cells must be run **in order, from top to bottom**: a cell often uses variables created by the cells above.
- If something looks strange, or you changed many things, use **Kernel → Restart Kernel and Run All Cells**. This is the single most useful trick: it tells you whether your notebook really works from scratch.
- The *last line* of a cell is printed automatically. To print anything else, use `print(...)`.
- `adata_sc` is a **variable** (a name that points to an object in memory). `adata_sc.n_obs` is an **attribute** of that object, `adata_sc.copy()` is a **method** (something it can do; notice the parentheses).
- Lines starting with `#` are comments: Python ignores them, they are there for humans.
- An error message looks scary but it is useful: **read the last line first**, it usually says what is wrong.

## Roadmap of the practical

| Part | Topic | The question we ask |
|---|---|---|
| 1 | Setup | What do we need before starting? |
| 2 | Where the data comes from | What is a single-cell count matrix, physically? |
| 3 | Loading the data | What is an `AnnData` object? |
| 4 | Quality control (QC) | Which cells are good cells? |
| 5 | Doublets | Which "cells" are actually two cells? |
| 6 | Normalization | How do we compare cells sequenced at different depth? |
| 7 | Highly variable genes | Which genes carry the interesting information? |
| 8 | PCA | How do we summarize 20 000 genes into ~30 numbers? |
| 9 | Neighbors graph and UMAP | How do we *see* the data? |
| 10 | Batch effects | Is the difference between samples technical or biological? |
| 11 | Clustering | How many groups of cells are there? |
| 12 | Marker genes | What makes each group different? |
| 13 | Cell type annotation | What biological cell type is each group? |
| 14 | Validation | Can we trust our annotation? |
| 15 | Cell states and composition | How does tissue composition change during healing? |
| 16 | Take-home messages | What did we learn, and what comes next? |

The pipeline in one picture:

```
 raw counts ──► QC filters ──► doublet removal ──► normalization (+ log) ──► highly variable genes
                                                                                      │
 cell types ◄── markers ◄── clusters (Leiden) ◄── UMAP ◄── neighbors graph ◄── PCA ◄──┘
```

Everything we compute is saved in `data/processed_data/SupBioTech`, so that you can always reload a checkpoint instead of recomputing from the beginning.

## Part 1 · Setup

> **Concept** · A Python notebook starts by *importing libraries*: collections of ready-made tools that other people wrote. We use:
>
> - **numpy** and **pandas**: numbers and tables;
> - **matplotlib** and **seaborn**: figures;
> - **scanpy**: the main library for single-cell analysis;
> - **anndata**: the data structure used by scanpy to store a dataset;
> - **harmonypy**: a batch correction method (Part 10).
>
> We also fix a **random seed**. Several steps of the analysis (PCA, UMAP, clustering, doublet simulation) use random numbers. Fixing the seed means that running the notebook twice gives the same result, which is essential for reproducible science.

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata as ad
import harmonypy

# --- general settings ---------------------------------------------------------
SEED = 0
np.random.seed(SEED)
sc.settings.verbosity = 1                      # 0 = silent ... 3 = very talkative
sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(5, 4))
sns.set_style("whitegrid")

# --- paths ---------------------------------------------------------------------
BASE = '/Users/sergio/Spatiotemporal_roadmap_of_human_skin_wound_healing'
SC_DATA = f'{BASE}/data/GSE241132_RAW'                   # raw data (input)
OUT = f'{BASE}/data/processed_data/SupBioTech'           # our results (output)
os.makedirs(OUT, exist_ok=True)

sc.logging.print_header()                      # versions of the libraries in use

> **Question** · Why do we print the versions of the libraries?
>
> <details><summary>Show answer</summary>
> Because results can change between versions (a default parameter changes, a bug is fixed). A reproducible analysis always records the software versions: if you cannot reproduce a result in two years, the first thing to compare is the versions.
> </details>

## Part 2 · Where does the data come from?

Before touching any number, we must understand how the numbers were produced. Most mistakes in scRNA-seq analysis come from forgetting what the data physically is.

### The biological question

The skin is wounded and then heals over weeks. Which cells are present at each stage? Which ones appear (e.g. immune cells at the beginning), disappear, or change their gene expression (e.g. keratinocytes migrating to close the wound)? The authors collected skin biopsies from the same donors at four moments:

| Sample name | Meaning |
|---|---|
| **D0** | day 0: intact, unwounded skin (the reference) |
| **D1** | 1 day after wounding |
| **D7** | 7 days after wounding |
| **D30** | 30 days after wounding |

The whole study has 3 donors × 4 timepoints = 12 samples and 58 823 cells. **In this practical we use one donor, PWH27** (4 samples), to keep computation times short.

### From tissue to a table of numbers (10x Genomics Chromium)

> **Concept** · The tissue is dissociated into a suspension of single cells. The suspension is loaded on a microfluidic chip that encapsulates, in principle, **one cell + one gel bead** in a tiny oil droplet. Inside the droplet:
>
> 1. the cell is lysed and its **messenger RNAs** (mRNAs) are released;
> 2. each gel bead carries millions of short DNA oligos, all containing (i) a **cell barcode**, identical for all oligos of that bead, so it identifies the droplet/cell; (ii) a **UMI** (Unique Molecular Identifier), a random sequence that is different for each oligo; (iii) a poly-dT tail that captures the poly-A tail of mRNAs;
> 3. each captured mRNA is reverse-transcribed into cDNA that carries the barcode of its cell and the UMI of its molecule;
> 4. all droplets are pooled and sequenced;
> 5. software (Cell Ranger) reads the sequences, aligns them to the genome, and for each (cell barcode, gene) counts the number of **distinct UMIs**.
>
> Counting UMIs, rather than reads, removes the amplification bias: if one molecule is copied 50 times by PCR, it has 50 reads but still **one** UMI.

The result is a matrix with **one row per cell and one column per gene**, where each entry is the number of mRNA molecules of that gene detected in that cell.

### What this implies (keep these in mind for the whole practical)

- **Only a small fraction of the mRNA molecules of a cell are captured and detected.** A typical cell contains hundreds of thousands of mRNA molecules, but we observe only a fraction of them (tens of thousands of UMIs at most, often much fewer). We see a *sample* of its transcriptome, not the whole.
- Consequently the matrix is **extremely sparse**: most entries are 0. A zero means *"no molecule of this gene was detected in this cell"*, which can mean the gene is really not expressed *or* that we simply missed its molecules (especially if the gene is lowly expressed).
- Cells differ in how many molecules were captured (cell size, droplet efficiency, sequencing depth). This **technical** variation in the total counts is not biology, and we will have to deal with it (Part 6).
- Some droplets contain **no cell** (only ambient RNA released by dying cells) and some contain **two cells** (doublets). Both must be identified and removed (Parts 4 and 5).
- Only mRNAs captured by their poly-A tail are seen: no small RNAs, and only the 3' end of each transcript is sequenced.

> **Take-home** · A scRNA-seq dataset is a very sparse table of *molecule counts*, obtained by sampling a small fraction of each cell's RNA. Almost every analysis step exists to cope with this fact.

> **Question** · A gene has a count of 0 in a cell. List two different explanations.
>
> <details><summary>Show answer</summary>
> (1) The gene is truly not expressed in that cell (e.g. a keratin gene in a fibroblast). (2) The gene is expressed, but at a low level, and none of its few mRNA molecules happened to be captured and sequenced (technical "dropout"). From a single zero we cannot know which one it is; this is why single-cell methods rely on many cells and on groups of genes, not on one gene in one cell.
> </details>

## Part 3 · Loading the data and meeting `AnnData`

> **Concept** · Cell Ranger delivers each sample as three files in a folder (the "10x mtx" format):
>
> - `matrix.mtx.gz`: the counts, stored as a *sparse* matrix (only the non-zero entries are written: position + value);
> - `barcodes.tsv.gz`: the name of each cell (its barcode, a string of 16 letters + `-1`);
> - `features.tsv.gz`: the name of each gene (Ensembl ID, gene symbol, type).
>
> `sc.read_10x_mtx` reads the three files and builds an **`AnnData`** object (*annotated data*), the standard container of scanpy:
>
> | Slot | What it contains | Shape |
> |---|---|---|
> | `.X` | the main matrix (counts now; later normalized values) | cells × genes |
> | `.obs` | a table with **one row per cell** (sample, QC metrics, cluster, cell type...) | cells × annotations |
> | `.var` | a table with **one row per gene** (gene name, is-mitochondrial, highly variable...) | genes × annotations |
> | `.layers` | extra matrices with the same shape as `.X` (e.g. a backup of the raw counts) | cells × genes |
> | `.obsm` | per-cell *embeddings* (PCA, UMAP coordinates) | cells × k |
> | `.uns` | unstructured stuff (parameters, results of tests, colors) | - |
>
> Whenever you wonder where something is stored, think: *"is it a property of cells (obs), of genes (var), or a full matrix (layers)?"*

We read the four samples of donor PWH27 one by one, label each cell with its donor and sample, and then **concatenate** them in a single object.

In [ ]:
donor = "PWH27"
timepoints = ["D0", "D1", "D7", "D30"]

adatas_sc = {}
for tp in timepoints:
    a = sc.read_10x_mtx(f"{SC_DATA}/{donor}{tp}")     # reads matrix + barcodes + features
    a.obs["donor"] = donor
    a.obs["sample"] = tp
    a.obs_names = [f"{bc}-{tp}" for bc in a.obs_names]  # barcodes can repeat across samples: make them unique
    adatas_sc[tp] = a
    print(f"{donor}{tp}: {a.n_obs} cells x {a.n_vars} genes")

adata_sc = ad.concat(list(adatas_sc.values()), join="inner")     # stack the cells of the 4 samples
adata_sc.obs["sample"] = pd.Categorical(adata_sc.obs["sample"], categories=timepoints)

import gc
del adatas_sc, a          # free memory: the 4 separate objects are no longer needed
gc.collect()
adata_sc

> **Concept** · Two details in the cell above are worth understanding.
>
> - **Unique cell names.** The barcode is a random 16-letter sequence. The same barcode can appear in two different samples (by chance, or because the same list of barcodes is used in every run). We append the timepoint (`-D0`, `-D1`...) so that every cell has a unique name.
> - **Memory.** Single-cell objects are big. After concatenating we delete the four separate objects (`del`) because they are now duplicated inside `adata_sc`. If your computer has little RAM and the kernel "dies" in the next steps (typically at the doublet step), this is the first thing to check: close other programs and, if needed, ask for a machine with more memory (8 GB is the minimum for this donor, 16 GB is comfortable).
> - **`join="inner"`.** When stacking datasets we keep only the genes present in *all* of them. Here all samples were aligned to the same reference, so we lose nothing.

### Look at the raw material

Let's see what the count matrix really looks like. We pick 10 cells and the 10 genes with the highest total counts.

In [ ]:
gene_totals = np.asarray(adata_sc.X.sum(axis=0)).ravel()
top_idx = gene_totals.argsort()[::-1][:10]

pd.DataFrame(
    adata_sc.X[:10, top_idx].toarray().astype(int),
    index=adata_sc.obs_names[:10],
    columns=adata_sc.var_names[top_idx],
)

Notice: even for the **most** abundant genes, many entries are small numbers (0, 1, 2, 3...). The values are *integers* because they are counts of molecules. Now let's quantify how sparse the full matrix is.

In [ ]:
n_cells, n_genes = adata_sc.shape
n_nonzero = adata_sc.X.nnz
print(f"Matrix: {n_cells} cells x {n_genes} genes = {n_cells * n_genes:,} entries")
print(f"Non-zero entries: {n_nonzero:,}  ({100 * n_nonzero / (n_cells * n_genes):.1f}%)")
print(f"Zeros: {100 * (1 - n_nonzero / (n_cells * n_genes)):.1f}%")

# memory: sparse vs dense
sparse_mb = (adata_sc.X.data.nbytes + adata_sc.X.indices.nbytes + adata_sc.X.indptr.nbytes) / 1e6
dense_mb = n_cells * n_genes * 4 / 1e6
print(f"\nMemory as sparse matrix: ~{sparse_mb:,.0f} MB   |   as dense matrix: ~{dense_mb:,.0f} MB")

> **Question** · Why is it so important that scanpy stores the matrix in *sparse* format?
>
> <details><summary>Show answer</summary>
> Because about 90% of the entries are zeros (compare the sparse and dense memory printed above). A dense matrix would store (and multiply, and copy) all those zeros: several GB of RAM for a single donor, tens of GB for the full study. The sparse format stores only the non-zero values and their positions, so memory and computation time are proportional to the number of detected molecules, not to cells × genes.
> </details>

### The same observation, from the point of view of cells and genes

Two distributions summarize the sparsity: how many genes are detected per cell, and in how many cells each gene is detected.

In [ ]:
genes_per_cell = adata_sc.X.getnnz(axis=1)       # number of non-zero entries per cell (= genes detected)
cells_per_gene = adata_sc.X.getnnz(axis=0)       # number of non-zero entries per gene (= cells where it is detected)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(genes_per_cell, bins=60, color="steelblue")
axes[0].set_xlabel("genes detected in the cell"); axes[0].set_ylabel("number of cells")
axes[0].set_title("Genes per cell")
axes[1].hist(np.log10(cells_per_gene + 1), bins=60, color="darkorange")
axes[1].set_xlabel("log10(1 + number of cells where the gene is detected)"); axes[1].set_ylabel("number of genes")
axes[1].set_title("Cells per gene")
fig.tight_layout(); plt.show()

print(f"Median genes detected per cell: {np.median(genes_per_cell):.0f}  (out of {n_genes})")
print(f"Genes never detected in any cell: {(cells_per_gene == 0).sum()}")
print(f"Genes detected in fewer than 10 cells: {(cells_per_gene < 10).sum()}")

> **Take-home** · A cell typically shows a few thousand of the ~36 000 genes of the reference (see the median printed above), and many genes are never (or almost never) seen. Genes detected in just a handful of cells contain essentially no usable information: we will remove them later.

## Part 4 · Quality control (QC)

> **Concept** · Not every barcode in the matrix is a good cell. Before any biological analysis we remove the barcodes that are *technical artefacts*. We compute three simple QC metrics **per cell**:
>
> | Metric | Meaning | What a problem looks like |
> |---|---|---|
> | `total_counts` | total number of UMIs in the cell (the *library size*) | very **low**: empty droplet or poor-quality cell; very **high**: possible doublet |
> | `n_genes_by_counts` | number of different genes detected | very **low**: empty droplet / dying cell; very **high**: possible doublet |
> | `pct_counts_mt` | % of the UMIs coming from **mitochondrial genes** (names starting with `MT-`) | **high**: stressed or dying cell |
>
> **Why mitochondrial reads reveal dying cells.** In a healthy cell, most mRNA is in the cytoplasm. If the cell membrane is damaged during tissue dissociation, the cytoplasmic mRNAs leak out and are lost, while the mRNAs inside the mitochondria (protected by the mitochondrial membranes) are retained. The fraction of mitochondrial reads therefore increases. This reasoning is specific to *dissociated single cells*; in spatial transcriptomics on intact tissue the same percentage means something different.

In [ ]:
adata_sc.var["mt"] = adata_sc.var_names.str.startswith("MT-")      # flag the mitochondrial genes
print("Mitochondrial genes found:", adata_sc.var_names[adata_sc.var["mt"]].tolist())

sc.pp.calculate_qc_metrics(adata_sc, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)

METRICS = ["total_counts", "n_genes_by_counts", "pct_counts_mt"]
TITLES = {
    "total_counts": "Total UMIs per cell",
    "n_genes_by_counts": "Detected genes per cell",
    "pct_counts_mt": "% mitochondrial counts per cell",
}
SC_SAMPLES = timepoints

# the new columns are in .obs: one value per cell
adata_sc.obs[METRICS].head()

### Look at the metrics, sample by sample

We start with **violin plots**: the width of the "violin" at a given height is proportional to the number of cells having that value; the box inside shows the median and the quartiles; each black dot is a cell. We look at one sample at a time, because samples can have different quality.

In [ ]:
for s in SC_SAMPLES:
    a = adata_sc[adata_sc.obs["sample"] == s]
    fig, axes = plt.subplots(1, len(METRICS), figsize=(12, 4))
    for ax, metric in zip(axes, METRICS):
        sns.violinplot(y=a.obs[metric], ax=ax, inner="box", cut=0)
        sns.stripplot(y=a.obs[metric], ax=ax, color="black", size=1.5, alpha=0.3)
        ax.set_title(TITLES[metric])
        ax.set_ylabel("")
    fig.suptitle(f"QC per sample: {s} (n={a.n_obs} cells)")
    fig.tight_layout()
    plt.show()

And now the four samples side by side, plus a table of medians.

In [ ]:
fig, axes = plt.subplots(1, len(METRICS), figsize=(14, 4))
for ax, metric in zip(axes, METRICS):
    sns.violinplot(
        data=adata_sc.obs, x="sample", y=metric, order=SC_SAMPLES,
        ax=ax, inner="box", cut=0,
    )
    ax.set_title(TITLES[metric])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="x", rotation=30)
fig.suptitle("QC combined across samples (donor PWH27)")
fig.tight_layout()
plt.show()

summary = adata_sc.obs.groupby("sample", observed=True)[METRICS].median().loc[SC_SAMPLES]
summary.insert(0, "n_cells", adata_sc.obs["sample"].value_counts().loc[SC_SAMPLES])
summary.round(1)

> **Question** · Do the four samples have the same number of cells and a similar quality? Is a difference between samples necessarily a problem?
>
> <details><summary>Show answer</summary>
> The number of cells and the medians differ between samples. This is normal: the number of captured cells depends on the biopsy and the loading, and the composition of the tissue changes during healing (e.g. immune cells are small cells with few mRNAs; inflamed wounds contain many of them). A difference between samples is not automatically a problem: we have to ask whether it is technical or biological. This is exactly the question we will ask again in Part 10.
> </details>

### Look at the metrics *together*

Looking at one metric at a time can hide patterns. Let's plot the metrics against each other: every dot is one cell.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
o = adata_sc.obs
pts = axes[0].scatter(o["total_counts"], o["n_genes_by_counts"], c=o["pct_counts_mt"], s=2, cmap="viridis", rasterized=True)
fig.colorbar(pts, ax=axes[0], label="% mito")
axes[0].set_xlabel("total_counts"); axes[0].set_ylabel("n_genes_by_counts")
axes[0].set_title("genes vs UMIs, colored by % mito")
axes[1].scatter(o["total_counts"], o["pct_counts_mt"], s=2, c="gray", alpha=0.5, rasterized=True)
axes[1].set_xlabel("total_counts"); axes[1].set_ylabel("pct_counts_mt")
axes[1].set_title("% mito vs UMIs")
sns.histplot(data=o, x="total_counts", hue="sample", log_scale=True, element="step",
             stat="density", common_norm=False, ax=axes[2])
axes[2].set_title("library size, per sample (log scale)")
fig.tight_layout(); plt.show()

How to read these plots:

- **Left.** Genes and UMIs grow together (more molecules → more different genes detected), and the curve bends because of saturation: after a while, extra UMIs come from genes already detected. Cells that fall *below* the main curve (many UMIs but few genes) have their counts concentrated on a few genes: typically dying cells or ambient RNA. The color shows that high-mito cells tend to be among the poor-quality ones.
- **Center.** Cells with very high % mito are usually cells with few UMIs: the cytoplasm has been lost.
- **Right.** The distribution of library sizes is wide: cells differ by several-fold, and extreme cells by orders of magnitude. This is a first hint of what we will have to normalize.

### Choosing thresholds: a decision, not a rule

> **Concept** · There is no universal threshold. Thresholds are a trade-off:
>
> - too **permissive**: low-quality barcodes and empty droplets stay in the data and form artificial clusters;
> - too **strict**: we throw away real cells, especially real cell types with naturally few mRNAs (for example some immune cells), and we bias the composition of the dataset.
>
> A good practice is to look at how many cells each threshold removes *before* deciding. Let's do it.

In [ ]:
# What fraction of cells (%) would be removed by each minimum number of genes?
sweep_genes = {}
for t in [200, 300, 500, 750, 1000]:
    sweep_genes[f"< {t} genes"] = adata_sc.obs.groupby("sample", observed=True)["n_genes_by_counts"].apply(lambda x: 100 * (x < t).mean())
print("% of cells removed, depending on the minimum number of genes")
display(pd.DataFrame(sweep_genes).loc[SC_SAMPLES].round(1))

sweep_mt = {}
for t in [5, 10, 20, 30]:
    sweep_mt[f"> {t}% mito"] = adata_sc.obs.groupby("sample", observed=True)["pct_counts_mt"].apply(lambda x: 100 * (x > t).mean())
print("\n% of cells removed, depending on the maximum mitochondrial percentage")
display(pd.DataFrame(sweep_mt).loc[SC_SAMPLES].round(1))

> **Question** · Look at the first table. Is the effect of the threshold the same in the four samples? Why would a *single* threshold applied blindly be risky?
>
> <details><summary>Show answer</summary>
> The percentage of removed cells differs between samples, because the samples have different depths and different quality. A threshold that is reasonable for one sample may eliminate a large part of another. That is why we always inspect the distributions per sample, and why we prefer thresholds that are "reasonable and conservative" and justified by the literature or by the data.
> </details>

### The thresholds of the original paper

In this practical we use **the same thresholds as the authors** (Methods, *"Human scRNA-seq data processing"*), so that our results can be compared with theirs:

- remove cells with **fewer than 500 genes**;
- remove cells with **fewer than 1 000 total UMIs**;
- remove cells with **more than 20% mitochondrial counts**.

First, how many cells fail each criterion? The criteria overlap (a cell with few UMIs also has few genes), so the sum is larger than the number of cells actually removed.

In [ ]:
fail = pd.DataFrame({
    "< 500 genes":   adata_sc.obs["n_genes_by_counts"] < 500,
    "< 1000 UMIs":   adata_sc.obs["total_counts"] < 1000,
    ">= 20% mito":   adata_sc.obs["pct_counts_mt"] >= 20,
})
print("Cells failing each criterion:")
print(fail.sum().to_string())
print(f"\nCells failing at least one criterion: {fail.any(axis=1).sum()} / {adata_sc.n_obs}"
      f"  ({100 * fail.any(axis=1).mean():.1f}%)")

Now we apply the filters, one by one, printing how many cells survive each step.

In [ ]:
n0 = adata_sc.n_obs

sc.pp.filter_cells(adata_sc, min_genes=500)       # keep cells with at least 500 detected genes
print(f"after min_genes=500   : {adata_sc.n_obs} cells")

sc.pp.filter_cells(adata_sc, min_counts=1000)     # keep cells with at least 1000 UMIs
print(f"after min_counts=1000 : {adata_sc.n_obs} cells")

adata_sc = adata_sc[adata_sc.obs["pct_counts_mt"] < 20].copy()    # keep cells with < 20% mitochondrial UMIs
print(f"after pct_mt < 20     : {adata_sc.n_obs} cells")

print(f"\nTotal removed: {n0 - adata_sc.n_obs} cells ({100 * (n0 - adata_sc.n_obs) / n0:.1f}%)")
adata_sc.obs["sample"].value_counts().loc[SC_SAMPLES]

> **Concept · why a copy?** In `adata_sc[mask].copy()` the brackets create a *view* (a window on the original data) and `.copy()` turns it into an independent object. Without `.copy()` scanpy would warn us and later modifications could behave unexpectedly.

> **Take-home** · QC removes barcodes that are not good cells: empty droplets and damaged cells (few genes/UMIs, high mitochondrial fraction). Thresholds are choices; we make them after looking at the distributions, and we report them.

## Part 5 · Doublets

> **Concept** · The microfluidic system loads cells randomly. Sometimes **two (or more) cells end up in the same droplet** and share the same barcode: this is a *doublet*. The resulting profile is a mixture of two cells, for example a keratinocyte plus a fibroblast. If we keep it, it can look like a "new" cell type with the markers of both, or it can bridge two real clusters in the UMAP and mislead the interpretation.
>
> The probability of a doublet grows with the number of loaded cells (a few % in a typical experiment). We cannot see them by eye, so we use a computational method.
>
> **How Scrublet works (idea):**
>
> 1. take random pairs of real cells and **sum their counts**: these are *simulated doublets*, for which we know the ground truth;
> 2. project the real cells and the simulated doublets in the same low-dimensional space (PCA);
> 3. for each real cell, ask: *among its nearest neighbors, what is the fraction of simulated doublets?* This is the **doublet score** (0–1);
> 4. choose a threshold on the score that separates two modes of the distribution, and call "predicted doublet" the cells above it.
>
> Doublets are detected **per sample** (`batch_key="sample"`), because doublets only happen *within* the same droplet run: the simulated doublets must be built from cells of the same sample.
>
> The paper used two doublet detectors (Scrublet and DoubletFinder) and removed the consensus. We use Scrublet only, since it is directly available in scanpy.

In [ ]:
sc.pp.scrublet(adata_sc, batch_key="sample")       # needs RAW counts in .X (they still are)
adata_sc.obs["predicted_doublet"].value_counts()

Let's look at the score distribution: a good run shows a large mode of real singlets with low scores and a small mode of high scores.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=adata_sc.obs, x="doublet_score", hue="predicted_doublet", bins=60,
             multiple="stack", ax=axes[0])
axes[0].set_title("Scrublet doublet score")
rate = adata_sc.obs.groupby("sample", observed=True)["predicted_doublet"].mean().loc[SC_SAMPLES] * 100
rate.plot.bar(ax=axes[1], color="slategray")
axes[1].set_ylabel("% predicted doublets"); axes[1].set_xlabel("")
axes[1].set_title("Predicted doublet rate per sample")
fig.tight_layout(); plt.show()

### See the doublets

Doublets should be cells with *more* counts and genes than average (they contain two cells' worth of RNA). Let's check.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric in zip(axes, ["total_counts", "n_genes_by_counts"]):
    sns.boxplot(data=adata_sc.obs, x="predicted_doublet", y=metric, ax=ax, showfliers=False)
    ax.set_title(TITLES[metric])
fig.tight_layout(); plt.show()

> **Question** · Doublets between two cells of the **same** type (two keratinocytes) are also formed. Will Scrublet detect them well? What does this tell us about the limits of the method?
>
> <details><summary>Show answer</summary>
> Not well: a "keratinocyte + keratinocyte" doublet looks like a keratinocyte with a bit more RNA, and the simulated doublets of the same type fall on top of the real cells of that type. Scrublet is most sensitive to doublets between *different* cell types (heterotypic), which are the ones that create artefactual clusters. Any computational doublet detection misses a fraction of the doublets and can remove a few real cells: it reduces the problem, it does not eliminate it.
> </details>

We now remove the predicted doublets.

In [ ]:
n_before = adata_sc.n_obs
adata_sc = adata_sc[~adata_sc.obs["predicted_doublet"]].copy()
print(f"Doublet removal: {n_before} -> {adata_sc.n_obs} cells")

### Cleaning the genes

Until now we filtered cells. We also filter **genes**, for two reasons: genes carrying no information cost computation and add noise, and some genes carry *technical* or *non-informative* signal. Following the paper:

1. **mitochondrial genes** (`MT-*`): they were useful to measure quality, but they mostly reflect cell stress and not cell identity;
2. **hemoglobin genes** (`HBA1, HBA2, HBB...`): they come from red blood cells or from blood contaminating the biopsy;
3. **genes detected in fewer than 10 cells**: too rare to be informative or to be tested.

In addition we remove **MALAT1**, a very abundant nuclear long non-coding RNA, which is often among the top genes in every cell and is commonly removed because it mostly reflects RNA quality. *Note: the paper mentions removing MALAT1 for the spatial data, not explicitly for scRNA-seq; here we remove it because it is a common practice, and we say so explicitly.*

The hemoglobin genes are listed **by name**. A pattern such as `HB*` would also catch `HBEGF`, a growth factor that is highly relevant for wound healing and is not hemoglobin!

In [ ]:
hb_genes = ["HBA1", "HBA2", "HBB", "HBD", "HBE1", "HBG1", "HBG2", "HBM", "HBQ1", "HBZ"]
mt_genes = adata_sc.var_names[adata_sc.var_names.str.startswith("MT-")].tolist()
remove = set(["MALAT1"] + mt_genes + hb_genes) & set(adata_sc.var_names)
print(f"Genes removed by name: {len(remove)}")

n_genes_before = adata_sc.n_vars
adata_sc = adata_sc[:, ~adata_sc.var_names.isin(remove)].copy()

sc.pp.filter_genes(adata_sc, min_cells=10)         # keep genes detected in >= 10 cells
print(f"Genes: {n_genes_before} -> {adata_sc.n_vars}")
print(f"Final object: {adata_sc.n_obs} cells x {adata_sc.n_vars} genes")
print("HBEGF still there:", "HBEGF" in adata_sc.var_names)

> **Take-home** · After QC, doublet removal and gene filtering, we have a table of *good* cells and *informative* genes. Still raw counts, though: next we have to make cells comparable.

## Part 6 · Normalization

> **Concept** · Two cells of the same type can have very different total counts only because more or fewer of their mRNAs were captured (technical variation). If we compare raw counts, **depth** dominates over biology. Normalization removes this effect.
>
> The classic approach has two steps:
>
> 1. **Library-size normalization** (`sc.pp.normalize_total`): divide each cell's counts by the cell's total, and multiply by a constant (here 10 000, "counts per 10 000"). After that every cell has the same total, and each value is the *relative* abundance of a gene in the cell.
> 2. **Log transformation** (`sc.pp.log1p`): apply $\log(1 + x)$. This compresses the huge dynamic range (a few genes with hundreds of counts, thousands with 0–3) so that a handful of very high genes do not dominate the analysis, and it makes the differences closer to *fold changes*, which is what matters biologically (a gene going from 1 to 2 is as relevant as from 50 to 100). The "+1" keeps zeros at zero (since $\log 1 = 0$).
>
> **Important:** we first **keep a copy of the raw counts** in a layer. Several later methods (and any statistical model of counts) need raw counts, not normalized values.

### A toy example to see why we normalize

Three cells, four genes. `cell_1` and `cell_2` are the **same cell type**; `cell_2` simply has 4 times more captured molecules. `cell_3` is a **different** cell type.

In [ ]:
toy = pd.DataFrame(
    {"gene_A": [4, 16, 0], "gene_B": [0, 0, 8], "gene_C": [6, 24, 6], "gene_D": [10, 40, 6]},
    index=["cell_1 (type 1, shallow)", "cell_2 (type 1, 4x deeper)", "cell_3 (type 2, shallow)"],
)
toy_norm = toy.div(toy.sum(axis=1), axis=0) * 100          # counts per 100 (instead of 10 000)
toy_log = np.log1p(toy_norm)

print("RAW COUNTS"); display(toy)
print("NORMALIZED (counts per 100)"); display(toy_norm.round(1))
print("NORMALIZED + log1p"); display(toy_log.round(2))

def dist(df, i, j):
    return float(np.sqrt(((df.iloc[i] - df.iloc[j]) ** 2).sum()))

print("\nEuclidean distance between cells:")
print(f"  raw      : cell_1 vs cell_2 (SAME type) = {dist(toy, 0, 1):5.1f}   | cell_1 vs cell_3 (different type) = {dist(toy, 0, 2):5.1f}")
print(f"  log-norm : cell_1 vs cell_2 (SAME type) = {dist(toy_log, 0, 1):5.2f}   | cell_1 vs cell_3 (different type) = {dist(toy_log, 0, 2):5.2f}")

> **Question** · Compare the distances. What goes wrong with the raw counts, and how does normalization fix it?
>
> <details><summary>Show answer</summary>
> With raw counts, `cell_1` is much "closer" to the cell of a different type (`cell_3`) than to its own twin `cell_2`, only because `cell_2` has more molecules: the analysis would group cells by depth instead of by identity. After normalization `cell_1` and `cell_2` become identical (distance 0), and `cell_3` is correctly far away. This is exactly what would happen in a real dataset with clusters built on distances (PCA, neighbors, clustering).
> </details>

### Why the logarithm?

A figure is worth more than words: how does $\log(1+x)$ transform values?

In [ ]:
x = np.linspace(0, 100, 400)
fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.plot(x, x, label="no transformation", color="gray")
ax.plot(x, np.log1p(x), label="log(1 + x)", color="crimson")
ax.set_xlabel("normalized expression"); ax.set_ylabel("transformed value")
ax.legend(); ax.set_title("The logarithm compresses large values")
plt.show()

Going from 0 to 10 changes the log value by ~2.4; going from 90 to 100 changes it by only ~0.1. Low-to-medium expression changes matter much more than differences between very highly expressed genes.

### Now on the real data

In [ ]:
# 1) keep the raw counts
adata_sc.layers["counts"] = adata_sc.X.copy()

lib_before = np.asarray(adata_sc.X.sum(axis=1)).ravel()
print(f"Library size BEFORE: min = {lib_before.min():.0f}, median = {np.median(lib_before):.0f}, max = {lib_before.max():.0f}")

# 2) normalize to 10 000 counts per cell
sc.pp.normalize_total(adata_sc, target_sum=1e4)
lib_after = np.asarray(adata_sc.X.sum(axis=1)).ravel()
print(f"Library size AFTER : min = {lib_after.min():.0f}, median = {np.median(lib_after):.0f}, max = {lib_after.max():.0f}")

# 3) log transform
sc.pp.log1p(adata_sc)
print("Largest value in the matrix after log1p:", round(float(adata_sc.X.max()), 2))

Every cell now sums to 10 000 *before* the log, whatever its initial depth. Let's look at the effect on one gene.

In [ ]:
gene = "KRT14"          # a keratinocyte (basal) gene: change it and try another one!
raw_vals = np.asarray(adata_sc.layers["counts"][:, adata_sc.var_names.get_loc(gene)].todense()).ravel()
norm_vals = np.asarray(adata_sc.X[:, adata_sc.var_names.get_loc(gene)].todense()).ravel()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(raw_vals[raw_vals > 0], bins=40, color="steelblue")
axes[0].set_title(f"{gene}: raw counts (cells with >0)"); axes[0].set_xlabel("UMIs")
axes[1].hist(norm_vals[norm_vals > 0], bins=40, color="crimson")
axes[1].set_title(f"{gene}: log-normalized (cells with >0)"); axes[1].set_xlabel("log(1 + CP10K)")
fig.tight_layout(); plt.show()
print(f"{gene} detected in {(raw_vals > 0).mean() * 100:.1f}% of the cells")

> **Take-home** · Normalization makes cells comparable by removing the depth effect; the log reduces the dominance of very high values. We keep the raw counts in `layers["counts"]` for the methods that need them.
>
> **Caveat** · Dividing by the total assumes that cells have similar total RNA content. If one cell type has a gigantic expression of a few genes (e.g. keratins in keratinocytes) other genes look relatively lower. More sophisticated normalizations exist (e.g. Pearson residuals, scran, SCTransform, used by the paper); the simple log-normalization is still the most common starting point.

## Part 7 · Highly variable genes (HVG)

> **Concept** · We have ~20 000 genes, but most of them vary little between cells or vary only because of noise (housekeeping genes such as *ACTB* are expressed everywhere). The genes that **differ between cell types and states** are the informative ones. Keeping only them:
>
> - removes noise and speeds up all the following steps;
> - focuses the analysis on biological structure.
>
> **How it is done.** For every gene we measure its mean expression and its variability (dispersion). Variability grows with the mean for technical reasons, so we compare each gene to genes with a *similar mean* and keep those that vary **more than expected**. We keep the top 2 000 (`n_top_genes=2000`).
>
> **`batch_key="sample"`**: the selection is done separately in each sample and then combined. Genes that are variable *in every sample* are favored; this avoids picking genes that are variable only because of a technical difference in one sample.

In [ ]:
sc.pp.highly_variable_genes(adata_sc, n_top_genes=2000, batch_key="sample")
print("Selected genes:", int(adata_sc.var["highly_variable"].sum()))
adata_sc.var[["means", "dispersions_norm", "highly_variable", "highly_variable_nbatches"]].head()

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.8))
v = adata_sc.var
ax.scatter(v.loc[~v.highly_variable, "means"], v.loc[~v.highly_variable, "dispersions_norm"],
           s=3, c="lightgray", label="other genes")
ax.scatter(v.loc[v.highly_variable, "means"], v.loc[v.highly_variable, "dispersions_norm"],
           s=4, c="crimson", label="highly variable")
ax.set_xscale("log")
ax.set_xlabel("mean expression"); ax.set_ylabel("normalized dispersion (variability above expected)")
ax.legend(markerscale=3); ax.set_title("Highly variable genes")
plt.show()

top = v[v.highly_variable].sort_values("dispersions_norm", ascending=False).head(25).index.tolist()
print("Top 25 most variable genes:\n", ", ".join(top))
print("\nAre some classical markers among the 2000 selected genes?")
print({g: bool(v.loc[g, "highly_variable"]) for g in ["KRT14", "KRT5", "COL1A1", "PTPRC", "PECAM1", "MLANA", "ACTB", "GAPDH"] if g in v.index})

> **Question** · Look at the list of the most variable genes. What kind of genes do you recognize? Why are they variable?
>
> <details><summary>Show answer</summary>
> They are genes with a very *specific* expression: for instance the late cornified envelope genes (*LCE1A, LCE2B...*), *FLG2*, *KPRP* and *CRCT1* mark terminally differentiated keratinocytes; *GNLY, XCL2* mark cytotoxic lymphocytes; *CCL17* dendritic cells; *IGHA1, IGLC2* plasma cells; *CCL21* lymphatic endothelium. They are variable across cells precisely because they are on in a few cell types and off in all the others. Note that the ranking favors very specific genes: broadly useful markers such as *KRT14* or *COL1A1* are also selected, but they are less extreme in this particular ranking.
> </details>

### Break it · how many HVGs?

2 000 is a convention, not a law. What if we use much fewer, or all the genes? To keep the experiment fast we work on a **random subsample of 5 000 cells** (we do not modify `adata_sc`). We repeat normalization + HVG + PCA + UMAP for each setting and color the cells by three marker genes (a keratinocyte gene, an immune gene, a fibroblast gene).

In [ ]:
# --- helper functions for the experiments (they work on COPIES, adata_sc is not modified) ---
sub = sc.pp.subsample(adata_sc, n_obs=5000, random_state=SEED, copy=True)

def prepare(adata_in, n_top_genes=2000, n_comps=30):
    """normalize + log + HVG + PCA, starting from raw counts, on a copy"""
    t = adata_in.copy()
    t.uns.pop("log1p", None)                          # forget that the parent was already log-transformed
    t.X = t.layers["counts"].copy()
    sc.pp.normalize_total(t, target_sum=1e4)
    sc.pp.log1p(t)
    if n_top_genes is None:
        t.var["highly_variable"] = True               # use ALL genes
    else:
        sc.pp.highly_variable_genes(t, n_top_genes=n_top_genes, batch_key="sample")
    sc.tl.pca(t, n_comps=n_comps, random_state=SEED)
    return t

def embed(t, n_pcs=30, n_neighbors=15, seed=SEED, min_dist=0.5):
    """neighbors + UMAP on a prepared object; returns the 2D coordinates"""
    sc.pp.neighbors(t, n_pcs=n_pcs, n_neighbors=n_neighbors, random_state=SEED, key_added="tmp_nn")
    sc.tl.umap(t, neighbors_key="tmp_nn", random_state=seed, min_dist=min_dist)
    return t.obsm["X_umap"].copy()

def scatter_xy(ax, xy, values, title, cmap="viridis"):
    order = np.argsort(values)                           # draw high values on top
    ax.scatter(xy[order, 0], xy[order, 1], c=np.asarray(values)[order], s=3, cmap=cmap)
    ax.set_title(title, fontsize=10); ax.set_xticks([]); ax.set_yticks([])

check_genes = [g for g in ["KRT14", "PTPRC", "COL1A1"] if g in adata_sc.var_names]
print("Subsample:", sub.n_obs, "cells | genes to color:", check_genes)

In [ ]:
settings = [(100, "100 HVG"), (2000, "2000 HVG (our choice)"), (None, "ALL genes")]
fig, axes = plt.subplots(len(settings), len(check_genes), figsize=(4 * len(check_genes), 3.6 * len(settings)))
for i, (n_hvg, label) in enumerate(settings):
    t = prepare(sub, n_top_genes=n_hvg)
    xy = embed(t)
    for j, g in enumerate(check_genes):
        vals = np.asarray(t[:, g].X.todense()).ravel()
        scatter_xy(axes[i, j], xy, vals, f"{label} · {g}")
fig.tight_layout(); plt.show()

> **Question** · Compare the three rows. Which genes give clean, well-separated populations? What happens with very few genes? And with all the genes?
>
> <details><summary>Show answer</summary>
> With too few genes (100) some structure is lost or blurred: the genes selected cannot distinguish all populations. With all the genes the structure is generally still visible but noisier and slower to compute: thousands of non-informative genes add random variation. Around 1 000–3 000 HVGs is the usual compromise, and the result is fairly stable in that range. The lesson: this parameter matters less than people fear, but it is not irrelevant, and you should always check that the main biological structure does not depend on it.
> </details>

> **Take-home** · HVG selection keeps the genes that distinguish cells and discards the noise. From now on we work with the 2 000 HVGs.

## Part 8 · PCA: from 2 000 dimensions to ~30

> **Concept** · Each cell is now a point in a 2 000-dimensional space (one axis per HVG). We cannot visualize it, and distances in so many dimensions are noisy. **Principal Component Analysis (PCA)** finds new axes, the *principal components* (PCs), which are **combinations of genes**:
>
> - **PC1** is the direction along which cells vary the most;
> - **PC2** is the direction of maximum variability *among those perpendicular to PC1*; and so on.
>
> Every PC is a "gene program": the genes with large positive loading on PC1 go up together when PC1 goes up, the genes with large negative loading go down. The first PCs capture the strong biological differences (e.g. epithelial vs. immune vs. stromal cells); later PCs capture finer structure and, at the end, noise. Keeping the first ~30 PCs gives a compact, **denoised** representation.
>
> PCA is a *linear* method and it is deterministic (up to the sign of each PC): it is the backbone for everything that follows (neighbors, UMAP, clustering).
>
> **Note on scaling.** Many tutorials scale each gene to unit variance (`sc.pp.scale`) before the PCA. We do not: we apply PCA directly on log-normalized HVGs. Both choices are used in practice; without scaling, more highly expressed genes weigh more.

In [ ]:
sc.tl.pca(adata_sc, n_comps=30)
adata_sc.obsm["X_pca"].shape            # cells x 30 components

### How many PCs? The elbow plot

Each PC explains a fraction of the total variance. We plot this fraction (on a log scale) for the first 50 PCs and look for the "elbow": the point after which PCs add little. (To see 50 PCs we compute a temporary PCA on a copy.)

In [ ]:
X_hvg = adata_sc[:, adata_sc.var["highly_variable"]].X            # only the 2000 HVGs, as in the real PCA
_, _, var_ratio, _ = sc.pp.pca(X_hvg, n_comps=50, return_info=True)   # PCA with 50 components, just to look at the variance

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, 51), var_ratio, "o-", ms=4)
ax.axvline(30, color="crimson", ls="--", label="30 PCs (our choice)")
ax.set_yscale("log"); ax.set_xlabel("principal component"); ax.set_ylabel("fraction of variance explained")
ax.set_title("Elbow plot"); ax.legend()
plt.show()
print("Variance explained by PC1, PC2, PC3:", np.round(var_ratio[:3] * 100, 2), "%")
print("Variance explained by the first 30 PCs:", round(var_ratio[:30].sum() * 100, 1), "%")

The first PCs explain a visibly larger part of the variance and the curve flattens progressively: there is no sharp cut. The choice of 30 is a safe and common compromise (the paper used 40 on a larger dataset).

### What does a PC represent? Look at the genes

We can read each PC by listing the genes with the most extreme loadings. Let's do it for the first three PCs.

In [ ]:
loadings = pd.DataFrame(adata_sc.varm["PCs"][:, :3], index=adata_sc.var_names, columns=["PC1", "PC2", "PC3"])
loadings = loadings[adata_sc.var["highly_variable"].values]      # only HVGs enter the PCA
for pc in loadings.columns:
    pos = loadings[pc].sort_values(ascending=False).head(10).index.tolist()
    neg = loadings[pc].sort_values().head(10).index.tolist()
    print(f"{pc}  positive end: {', '.join(pos)}")
    print(f"{' ' * len(pc)}  negative end: {', '.join(neg)}\n")

> **Question** · Look at the genes at the two ends of each PC. What cell populations do they separate?
>
> <details><summary>Show answer</summary>
> The exact signs depend on the run, but typically PC1 opposes epithelial cells (keratins such as *KRT5/KRT14/KRT1*) to stromal/immune cells (e.g. collagens, *VIM*, immune genes); PC2 and PC3 separate further populations (for instance immune vs. fibroblasts vs. endothelial/mural cells). Each PC is a pattern of co-expressed genes that distinguishes major cell populations. This is why PCA is a good basis to find cell types.
> </details>

We can also **color the cells in PC space** by marker genes, to confirm that the first PCs already separate major populations.

In [ ]:
pca_genes = [g for g in ["KRT14", "PTPRC", "COL1A1"] if g in adata_sc.var_names]
sc.pl.pca(adata_sc, color=pca_genes, components=["1,2", "1,3"], ncols=3, cmap="viridis")

### Break it · how many PCs go in the next step?

The next steps use only the first `n_pcs` components. What if we use too few, or too many? Again on the subsample of 5 000 cells (compute a 50-PC PCA once, then try different values).

In [ ]:
base = prepare(sub, n_top_genes=2000, n_comps=50)
pc_values = [3, 10, 30, 50]
fig, axes = plt.subplots(len(check_genes), len(pc_values), figsize=(4 * len(pc_values), 3.6 * len(check_genes)))
for j, k in enumerate(pc_values):
    xy = embed(base, n_pcs=k)
    for i, g in enumerate(check_genes):
        vals = np.asarray(base[:, g].X.todense()).ravel()
        scatter_xy(axes[i, j], xy, vals, f"{k} PCs · {g}")
fig.tight_layout(); plt.show()

> **Question** · With 3 PCs, what is lost? Do we see a big difference between 30 and 50 PCs?
>
> <details><summary>Show answer</summary>
> With 3 PCs we only keep the strongest axes: the major populations are separated but finer ones are merged. Between 30 and 50 the picture is very similar: the added PCs carry little signal. Too few PCs lose real biology; too many add noise and computing time with no gain. When in doubt, take a number in the plateau of the elbow plot and verify the result is stable.
> </details>

> **Take-home** · PCA summarizes the data in ~30 "gene programs". It reduces noise and dimensionality, and it is the input of everything that follows.

## Part 9 · The neighbors graph and the UMAP

### The k-nearest-neighbors (kNN) graph

> **Concept** · In the 30-dimensional PCA space, cells with similar expression are close to each other. We build a **graph**: every cell is a node connected to its **k nearest neighbors** (`n_neighbors=15` by default). This graph is the real basis of the next two steps:
>
> - **clustering** (Part 11) looks for groups of cells that are densely connected with each other;
> - **UMAP** (below) places the nodes on a 2D plane so that connected cells stay close.
>
> The graph is built *once* and used by both; the UMAP is just a drawing of it.

### UMAP

> **Concept** · A UMAP (Uniform Manifold Approximation and Projection) is a way to draw in 2D the neighborhood structure of the data. Cells similar to each other end up close on the plot. It is very useful to explore and communicate, **but it has to be read with care**:
>
> - it preserves **local** neighborhoods well, while **global distances are not reliable**: the distance between two islands, or the shape and size of an island, do not have a quantitative meaning;
> - the plot changes with the parameters and with the random seed;
> - it is a *visualization*, not a statistical analysis. Clusters are computed on the graph, not on the UMAP.

In [ ]:
sc.pp.neighbors(adata_sc, n_pcs=30)          # builds the kNN graph (default: 15 neighbors), stored as "neighbors"
sc.tl.umap(adata_sc)                          # draws it in 2D (coordinates in adata_sc.obsm["X_umap"])
adata_sc.obsm["X_umap_uncorrected"] = adata_sc.obsm["X_umap"].copy()    # keep a safe copy of this UMAP

sc.pl.umap(adata_sc, color="sample", title="UMAP colored by sample (uncorrected)")

We can color the same UMAP by any variable: the QC metrics (to check that the structure is not driven by technical effects) and the expression of marker genes (to start giving a meaning to the islands).

In [ ]:
sc.pl.umap(adata_sc, color=["total_counts", "pct_counts_mt", "doublet_score"], ncols=3, cmap="viridis")

In [ ]:
umap_genes = [g for g in ["KRT14", "KRT1", "COL1A1", "PTPRC", "PECAM1", "MLANA", "TPSB2", "TAGLN"] if g in adata_sc.var_names]
sc.pl.umap(adata_sc, color=umap_genes, ncols=4, cmap="viridis")

> **Question** · Each island is dominated by a different marker. Which cell types do you recognize? Does the QC coloring (library size, mitochondrial %) look uniform over the islands, or are there islands with a strikingly different value? What would that suggest?
>
> <details><summary>Show answer</summary>
> *KRT14/KRT1* mark keratinocytes (basal/differentiated), *COL1A1* fibroblasts, *PTPRC* immune cells, *PECAM1* endothelial cells, *MLANA* melanocytes, *TPSB2* mast cells, *TAGLN* pericytes/smooth muscle. The QC metrics differ somewhat between cell types for biological reasons (cell size and RNA content are different: mast cells or lymphocytes naturally have fewer UMIs). An island with a very different value of a QC metric should be examined: it could be a real small cell type or a technical artefact (low-quality cells, or doublets with high scores).
> </details>

### Break it · the same data, different parameters

First, `n_neighbors`: the number of neighbors in the graph. Small values emphasize very local structure (fragmented islands), large values smooth it out.

In [ ]:
nn_values = [5, 15, 50, 150]
fig, axes = plt.subplots(1, len(nn_values), figsize=(4 * len(nn_values), 3.8))
g_color = check_genes[0]
vals = np.asarray(base[:, g_color].X.todense()).ravel()
for ax, k in zip(axes, nn_values):
    xy = embed(base, n_pcs=30, n_neighbors=k)
    scatter_xy(ax, xy, vals, f"n_neighbors = {k} · {g_color}")
fig.tight_layout(); plt.show()

Second, the **random seed**. UMAP starts from a random initial layout. We recompute it with three different seeds on exactly the same graph.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for ax, s in zip(axes, [0, 1, 2]):
    xy = embed(base, n_pcs=30, n_neighbors=15, seed=s)
    scatter_xy(ax, xy, vals, f"seed = {s} · {g_color}")
fig.tight_layout(); plt.show()

> **Question** · What changes and what stays the same across the seeds and the `n_neighbors` values? What can we conclude about reading distances on a UMAP?
>
> <details><summary>Show answer</summary>
> The *groups* are the same (the same cells stay together), but their position, orientation, shape and the distance between groups change with the seed and with the parameters. Hence: do not interpret the distance between two islands as "how different they are", do not read a position (up/down/left/right) as meaningful, and do not conclude that a population is "more heterogeneous" because its island is bigger. The UMAP is good for seeing *which cells group together*, not for measuring anything.
> </details>

Before moving on we free the memory used by the experiments (the subsampled copies are no longer needed).

We save a checkpoint of our processed object, so that we can come back here without redoing the previous steps.

In [ ]:
del sub, base, t                  # free memory
gc.collect()

adata_sc.write_h5ad(f"{OUT}/sc_data_PWH27_processed.h5ad")
print("saved:", f"{OUT}/sc_data_PWH27_processed.h5ad")
# To reload it later (in a new session), uncomment:
# adata_sc = sc.read_h5ad(f"{OUT}/sc_data_PWH27_processed.h5ad")

> **Take-home** · The kNN graph defines who is similar to whom; the UMAP is a 2D drawing of that graph. Use it to look, not to measure.

## Part 10 · Batch effects: should we "correct" them?

> **Concept** · A **batch effect** is a systematic difference between groups of cells due to *how* they were processed (different day, operator, reagent lot, sequencing run, donor) and not to biology. When samples are processed separately, the same cell type can end up in a different place in the UMAP in each sample. **Batch-correction methods** (Harmony, BBKNN, Scanorama, scVI...) try to align the samples so that the same cell type overlaps.
>
> But correcting is not always the right thing to do. The key question is: **what does my "batch" variable represent?**
>
> - If the batch is *purely technical* (e.g. the same kind of sample processed in two different runs), correcting it is appropriate.
> - If the batch is **confounded with the biology** you want to study, correcting it removes the signal of interest.
>
> In our dataset the variable `sample` is D0 / D1 / D7 / D30, from one donor, with **one sample per timepoint**. So `sample` is simultaneously the *technical batch* and the *biological condition* (time since wounding). There is no replicate of the same timepoint processed in a different batch that would allow the two effects to be separated.
>
> To make the point, we will apply Harmony *anyway*, on `sample`, and see what happens to the UMAP.
>
> **Harmony in one sentence:** it iteratively moves the cells in PCA space so that, in every small neighborhood, the cells of the different samples are mixed in proportion, while trying to keep cells of the same type together.

In [ ]:
# NOTE: sc.external.pp.harmony_integrate is currently broken with harmonypy >= 2.0 (shape mismatch),
# so we call harmonypy directly.
ho = harmonypy.run_harmony(adata_sc.obsm["X_pca"], adata_sc.obs, ["sample"])
Z = np.asarray(ho.Z_corr)
if Z.shape[0] != adata_sc.n_obs:      # older harmonypy versions return (n_pcs, n_cells)
    Z = Z.T
adata_sc.obsm["X_pca_harmony"] = Z

# a separate graph (key "harmony") and a separate UMAP: we do NOT overwrite the uncorrected ones
sc.pp.neighbors(adata_sc, use_rep="X_pca_harmony", key_added="harmony")
sc.tl.umap(adata_sc, neighbors_key="harmony")
adata_sc.obsm["X_umap_harmony"] = adata_sc.obsm["X_umap"].copy()
adata_sc.obsm["X_umap"] = adata_sc.obsm["X_umap_uncorrected"].copy()   # restore the uncorrected UMAP as the default

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc.pl.embedding(adata_sc, basis="X_umap_uncorrected", color="sample", ax=axes[0], show=False, title="uncorrected")
sc.pl.embedding(adata_sc, basis="X_umap_harmony", color="sample", ax=axes[1], show=False, title="Harmony (batch = sample)")
fig.tight_layout(); plt.show()

> **Concept · a technical warning for programmers.** Notice that we gave the new graph its own name (`key_added="harmony"`) and we stored each UMAP in its own slot (`X_umap_uncorrected`, `X_umap_harmony`). By default, `sc.pp.neighbors` and `sc.tl.umap` **overwrite** the previous results (`uns["neighbors"]`, `obsm["X_umap"]`) silently. Many analyses go wrong because of this: the plot you think is "uncorrected" is in reality the last UMAP you computed.

> **Question** · After Harmony the four samples are mixed. Is this a success? What biological information could we have just erased?
>
> <details><summary>Show answer</summary>
> The mixing looks "nice", but remember that the samples are not technical replicates: they are different moments of wound healing. The real biological differences between D0, D1, D7 and D30 (e.g. immune cells infiltrating at D1, keratinocytes in a wound-activated state at D7, remodeled fibroblasts at D30) can be partly removed by the correction, since the algorithm cannot distinguish "technical" from "biological". The correct strategy would be to correct for **donor** (technical/individual variation), in a dataset with several donors at each timepoint, and to verify that the timepoints are preserved. Here we simply don't have the design to do it, so the safest choice is to continue with the uncorrected data. **A batch-correction method does not know what is biology: you do.**
> </details>

**Decision.** For the rest of the practical we continue with the **uncorrected** PCA, graph and UMAP.

> **Take-home** · Always ask what your batch variable represents before correcting it. Mixed samples in a UMAP are not, by themselves, a sign of a good analysis.

## Part 11 · Clustering

> **Concept** · The cells form groups that correspond (roughly) to cell types and states. **Clustering** assigns every cell to a group, automatically. In single-cell analysis the standard approach works on the kNN graph (Part 9) and uses a **community detection** algorithm:
>
> - **Louvain** and **Leiden** look for groups of nodes that are densely connected among themselves and sparsely connected with the rest. **Leiden** is the improved successor of Louvain and guarantees that the communities it finds are internally connected; it is today's default. (The paper used Louvain.)
> - The key parameter is the **resolution**. A low resolution produces few, large clusters; a high resolution produces many small ones. **There is no "true" number of clusters**: cell types are organized in hierarchies (keratinocytes → basal / suprabasal / ...), so the question is always "at which level of detail do I want to look?".
>
> Clustering is *unsupervised*: the algorithm knows nothing of biology. Clusters are numbered by decreasing size and **numbers are arbitrary labels** (cluster 3 is not "more" than cluster 2). A cluster is *not yet* a cell type: interpreting it comes later (Parts 12–13).

### First: explore the resolution by hand

Before using any tool that chooses the resolution for us, let's see what it does. We run Leiden at six resolutions on the same graph and count the clusters.

In [ ]:
resolutions = [0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
for r in resolutions:
    sc.tl.leiden(adata_sc, neighbors_key="neighbors", resolution=r, key_added=f"leiden_r{r}",
                 flavor="igraph", directed=False)

n_clusters = pd.Series({r: adata_sc.obs[f"leiden_r{r}"].nunique() for r in resolutions})
print(n_clusters.to_string())

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(n_clusters.index, n_clusters.values, "o-")
ax.set_xscale("log"); ax.set_xlabel("resolution"); ax.set_ylabel("number of clusters")
ax.set_title("More resolution = more clusters")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, r in zip(axes.ravel(), resolutions):
    sc.pl.embedding(adata_sc, basis="X_umap", color=f"leiden_r{r}", ax=ax, show=False, legend_loc=None,
                    title=f"resolution {r}  ({n_clusters[r]} clusters)")
fig.tight_layout(); plt.show()

Are the clusters at a higher resolution a **subdivision** of those at a lower resolution? Let's cross-tabulate resolution 0.2 (columns) with resolution 1.0 (rows). Each row is a fine cluster: the cell with value 1 shows to which coarse cluster it belongs.

In [ ]:
ct = pd.crosstab(adata_sc.obs["leiden_r1.0"], adata_sc.obs["leiden_r0.2"], normalize="index")
fig, ax = plt.subplots(figsize=(8, 8))
sns.heatmap(ct, cmap="Blues", ax=ax, yticklabels=1, cbar_kws={"label": "fraction of the fine cluster"})
ax.set_xlabel("coarse clusters (resolution 0.2)"); ax.set_ylabel("fine clusters (resolution 1.0)")
plt.show()

> **Question** · What do you see? Is there a "right" resolution?
>
> <details><summary>Show answer</summary>
> The fine clusters are mostly nested inside the coarse ones (each row is dominated by one column): increasing the resolution splits clusters into sub-clusters. There is no single right answer: at low resolution we see the major cell types, at high resolution their sub-states and, if we go too far, we split a homogeneous population in arbitrary pieces. The right resolution depends on the question *and* on whether the sub-clusters are biologically meaningful (different marker genes). A statistical criterion can help: see below.
> </details>

### Choosing the resolution with RESOLUTE

> **Concept** · **RESOLUTE** is a tool that scans a range of resolutions and gives a score for each one. We use three independent criteria:
>
> - **Calinski–Harabasz**: ratio between the spread *between* clusters and the spread *within* them (high = compact and well-separated clusters);
> - **BIC** (Bayesian Information Criterion): quality of the fit of a model, penalized by the number of clusters (to avoid choosing a huge number of clusters);
> - **Stability**: repeating the clustering on random subsamples (bootstrap), how often do the cells stay together? (high = reproducible clusters).
>
> We run it on the **uncorrected PCA** (`use_rep="X_pca"`), in line with our decision in Part 10.

In [ ]:
try:
    import resolute as rs            # name of the package in older versions
except ImportError:
    import sc_resolute as rs         # name in the current version (pip install from GitHub)

results_calinski = rs.run_resolute(
    adata=adata_sc,
    score_value="calinski",
    use_rep="X_pca",      # uncorrected data, as decided
    min_res=0.1, max_res=2.0, step=0.1,
    clustering_algorithm="leiden",
)
best_resolution = results_calinski["best_score_res"]
print("Best resolution (Calinski-Harabasz):", best_resolution)

In [ ]:
results_bic = rs.run_resolute(
    adata=adata_sc,
    score_value="bic",
    use_rep="X_pca",
    compute_stability=False,       # we already computed the stability above
    min_res=0.1, max_res=2.0, step=0.1,
    clustering_algorithm="leiden",
)

### Reading the result

Two out of three criteria (Calinski–Harabasz and stability) agree on **resolution 0.2** as the optimal choice (17 clusters). The BIC instead keeps decreasing almost along the whole tested range (with only small bumps): it keeps rewarding models with more and more clusters, so its "deepest dip" at a high resolution is not a genuine optimum, just the consequence of a trend that does not stop. (With tens of thousands of cells, the data term of the BIC dominates the penalty for the number of clusters.) We therefore do not give the BIC the same weight as the other two criteria.

As a first pass we proceed with **resolution 0.2**, since it is supported by two independent and conceptually different criteria. We treat this as a *working choice*: the real test is whether the resulting clusters map onto coherent, biologically interpretable cell types when we look at their marker genes, not the resolution value itself.

> **Concept · replicate the whole configuration, not just a number.** The same numeric `resolution` gives different clusters if the algorithm is configured differently. RESOLUTE calls Leiden with `flavor="igraph", directed=False`; scanpy's default is different (`flavor="leidenalg"`). If we want to reproduce the RESOLUTE clustering we must use **exactly the same configuration**, as in the next cell. In general: when a tool selects a hyperparameter for you, reproduce all the settings of the underlying algorithm, not only the final value.

In [ ]:
# remove the exploratory clusterings, we don't need them anymore
adata_sc.obs = adata_sc.obs.drop(columns=[c for c in adata_sc.obs.columns if c.startswith("leiden_r")])
for k in [k for k in adata_sc.uns.keys() if k.startswith("leiden_r")]:
    del adata_sc.uns[k]

# Final clustering at the resolution selected by RESOLUTE
sc.tl.leiden(
    adata_sc,
    neighbors_key="neighbors",
    resolution=0.2,
    key_added="leiden",
    flavor="igraph",
    directed=False,
)
print(adata_sc.obs["leiden"].value_counts().to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sc.pl.embedding(adata_sc, basis="X_umap", color="sample", ax=axes[0], show=False, title="colored by sample")
sc.pl.embedding(adata_sc, basis="X_umap", color="leiden", ax=axes[1], show=False, legend_loc="on data",
                title="Leiden clusters (resolution=0.2)")
fig.tight_layout(); plt.show()

> **Question** · How many clusters do we obtain? Are they equally sized? Are all clusters present in all four samples?
>
> <details><summary>Show answer</summary>
> We obtain 17 clusters of very different sizes (a few large ones, several small ones). To answer the third question, cross-tabulate `leiden` and `sample`: some clusters are balanced across the timepoints, others are very unbalanced (e.g. immune clusters mostly in wounded samples). This will be the subject of Part 15.
> </details>

In [ ]:
cluster_by_sample = pd.crosstab(adata_sc.obs["leiden"], adata_sc.obs["sample"])[SC_SAMPLES]
cluster_by_sample.div(cluster_by_sample.sum(axis=1), axis=0).round(2)    # each row: fraction of the cluster in each sample

In [ ]:
# Save checkpoint: clustering
adata_sc.write_h5ad(f"{OUT}/adata_sc_leiden_res0.2.h5ad")

## Part 12 · Marker genes: what makes each cluster different?

> **Concept** · To understand what a cluster is, we look for the genes that distinguish it from all the others: its **marker genes**. The most common approach is a differential-expression test, **one cluster against the rest**. For every gene and every cluster, we compare the expression in the cells inside the cluster vs. outside. We use the **Wilcoxon rank-sum test**: it compares the *ranks* of the values in the two groups, so it does not assume any particular distribution (a good property, given how skewed single-cell data are).
>
> For each gene scanpy returns:
>
> | Column | What it tells us |
> |---|---|
> | `scores` | the test statistic (z-score): higher = more clearly higher in the cluster |
> | `logfoldchanges` | how much higher (log2) the mean expression is in the cluster than outside |
> | `pvals_adj` | p-value corrected for the multiple testing of thousands of genes |
> | `pct_nz_group` | % of cells **inside** the cluster expressing the gene |
> | `pct_nz_reference` | % of cells **outside** the cluster expressing the gene |
>
> A good marker has a large score and fold-change, is expressed in **most** cells of the cluster (high `pct_nz_group`) and in **few** cells outside (low `pct_nz_reference`).
>
> **Two warnings.**
>
> 1. *p-values are not very informative here.* With thousands of cells, nearly every difference is "significant". We use the ranking (scores, fold-changes, percentages) as a guide, not the p-value as a verdict.
> 2. *Circularity.* The clusters were defined using these same genes, so by construction they differ. The test tells us *which genes characterize* the clusters, not whether the clusters are "statistically real". The statistical proof of a cluster should come from an independent source (other data, other markers, experiments).

In [ ]:
sc.tl.rank_genes_groups(
    adata_sc, groupby="leiden", method="wilcoxon", key_added="rank_genes_leiden", pts=True
)

# Dotplot of the data-driven top marker genes per cluster
sc.tl.dendrogram(adata_sc, groupby="leiden", use_rep="X_pca")
sc.pl.rank_genes_groups_dotplot(
    adata_sc, groupby="leiden", key="rank_genes_leiden",
    n_genes=5, standard_scale="var",
)

> **Concept · how to read a dot plot.** Rows (here) are clusters and columns are genes. The **size** of a dot is the *percentage of cells* of the cluster expressing the gene; the **color** is the *mean expression* (scaled between 0 and 1 for each gene with `standard_scale="var"`). A good marker is a **big, dark dot in one row and nothing elsewhere**. A gene with big dots in all rows is not a marker. The dendrogram on the side groups similar clusters together.

It is also useful to read the markers as a table, with the numbers.

In [ ]:
top5 = (
    sc.get.rank_genes_groups_df(adata_sc, group=None, key="rank_genes_leiden")
    .sort_values("scores", ascending=False)
    .groupby("group", observed=True)
    .head(5)
).sort_values(by=["group", "logfoldchanges"], ascending=[True, False])
top5["group"] = top5["group"].astype(int)
top5 = top5.sort_values(by=["group", "logfoldchanges"], ascending=[True, False])
top5["pct_nz_group"] = (top5["pct_nz_group"] * 100).round(0)
top5["pct_nz_reference"] = (top5["pct_nz_reference"] * 100).round(0)
top5["logfoldchanges"] = top5["logfoldchanges"].round(2)
print(top5[["group", "names", "logfoldchanges", "pct_nz_group", "pct_nz_reference"]].to_string())

### Break it · does the test matter?

Do different statistical tests give the same markers? We compare the top 20 genes of the Wilcoxon test with the top 20 of a classical **t-test** for three clusters.

In [ ]:
sc.tl.rank_genes_groups(adata_sc, groupby="leiden", method="t-test", key_added="rank_genes_ttest")

for g in ["0", "3", "7"]:
    w = set(sc.get.rank_genes_groups_df(adata_sc, group=g, key="rank_genes_leiden").head(20)["names"])
    t = set(sc.get.rank_genes_groups_df(adata_sc, group=g, key="rank_genes_ttest").head(20)["names"])
    print(f"cluster {g}: {len(w & t)}/20 genes in common between Wilcoxon and t-test")

> **Question** · Do the two tests agree? What does this tell us about how much we should worry about the exact method?
>
> <details><summary>Show answer</summary>
> The overlap is usually substantial but not complete. For well-separated clusters, any reasonable test finds essentially the same strong markers; differences appear in the middle of the ranking, where genes are less specific. We should not over-interpret the exact rank of a gene, but we can trust the strongest and most specific markers.
> </details>

### Compare with the markers reported in the paper

The authors list marker genes for the major cell types. We use them as an independent reference (`paper_marker_genes.csv`, which contains only markers read directly from the text of the paper).

In [ ]:
marker_table = pd.read_csv(f"{BASE}/data/paper_marker_genes.csv")

# Build a marker dictionary usable with sc.pl.dotplot, for the major scRNA-seq cell types
sc_major_markers = {
    row["cell_type"]: row["marker_genes"].split(";")
    for _, row in marker_table.query("dataset == 'scRNA-seq' and granularity == 'major'").iterrows()
}
sc_major_markers = {k: [g for g in v if g in adata_sc.var_names] for k, v in sc_major_markers.items()}
print(sc_major_markers)

sc.pl.dotplot(adata_sc, sc_major_markers, groupby="leiden", standard_scale="var", dendrogram=True)

> **Take-home** · Marker genes are the bridge between clusters (a mathematical object) and cell types (a biological concept). We read the top genes of each cluster, look at their specificity (percentages), and compare them with the literature.

## Part 13 · Cell type annotation: from clusters to cell types

> **Concept** · Leiden gives a fine partition of the data (here 17 clusters) based only on transcriptional similarity. These clusters are not yet cell types. **Annotation** is a manual step: for each cluster we look at its marker genes (from the test above) and at known marker genes (from the literature) to decide which cell type it most likely represents.
>
> It is expected that **several clusters map to the same cell type** (different states of the same cell population: for instance basal and differentiated keratinocytes). This is a many-to-one relationship, and the paper itself follows the same logic: 27 fine-grained clusters (full dataset, Louvain at resolution 0.8) are grouped into a small number of major cell types, based on their marker genes.
>
> Annotation is **hypothesis-driven interpretation**, not a measurement: it requires biological knowledge, can be debated, and can be revised. For this reason we always (i) write down the evidence for each decision and (ii) validate it independently, as in Part 14.

### Our interpretation of each cluster

For each cluster we inspected its top differentially expressed genes (one-vs-rest Wilcoxon test) and matched them against known markers.

| Cluster | Top marker genes | Proposed cell type | Notes |
|---|---|---|---|
| 0 | KRT15, DST, COL17A1, MIR205HG | Keratinocytes | Basal state |
| 16 | S100A2, COL17A1, KRT5, KRT14 | Keratinocytes | Basal state |
| 11 | SPINK5, CALML5, SLURP1, SBSN | Keratinocytes | Differentiated / granular state |
| 2 | LGALS7, LGALS7B, KRT1, PERP | Keratinocytes | Suprabasal state |
| 6 | RPLP0, MT1X, KRT5, SNHG29 | Keratinocytes | Ribosomal / metallothionein-dominated signature. Possibly a lower-quality or stress-associated cluster rather than a distinct biological state |
| 15 | S100A14, MT1G, LY6D, S100A9 | Keratinocytes | S100A9 suggests a wound-activated state |
| 3 | HLA-DRB1, HLA-DRA, CD74, HLA-DPB1 | Myeloid cells | MHC-II high (dendritic-like) |
| 4 | FTL, CTSS, AIF1, CD68 | Myeloid cells | Macrophage-like |
| 10 | RPS27, B2M, RPL28, TMSB4X | Lymphoid cells (tentative) | Marker genes are largely non-specific / housekeeping. Needs a targeted verification (see below) |
| 12 | TPSB2, HPGD, TPSAB1, HPGDS | Mast cells | Unambiguous (tryptase genes) |
| 8 | FTH1, CFD, SERPING1, C1S | Fibroblasts | |
| 13 | VIM, LGALS1, TPM2, CAVIN3 | Fibroblasts | Possible myofibroblast-like state |
| 14 | SPARC, COL3A1, COL1A1, LUM | Fibroblasts | |
| 7 | MLANA, DCT, TYRP1 | Melanocytes | Unambiguous |
| 1 | MCAM, TAGLN, CALD1, MYL9 | Pericytes and smooth muscle cells | |
| 5 | MMRN1, CCL21, TFF3 | Endothelial cells | Lymphatic endothelial state |
| 9 | PECAM1, SPARCL1, GNG11 | Endothelial cells | Vascular state |

*The cluster numbers can differ if you rerun the notebook on a different machine or with different library versions: always check the marker genes, not only the number.*

**Note on provenance.** The top marker genes come from our own differential expression. The finer state descriptions (basal, differentiated, wound-activated...) are interpretations based on general knowledge of skin biology, not individually validated: read them as working hypotheses.

### A difficult case: cluster 10

The top genes of cluster 10 (*RPS27, B2M, RPL28, TMSB4X*) are ribosomal or housekeeping-like: they do not say what the cell is. When the data-driven markers are not informative, we **test specific hypotheses**: if these are T / NK cells, then *CD3D, CD3E* (T-cell receptor complex) and *NKG7* (cytotoxic granules) should be expressed in this cluster and not elsewhere.

In [ ]:
# Targeted check: T/NK cell markers
sc.pl.dotplot(adata_sc, {"T/NK markers": ["CD3D", "CD3E", "NKG7"]}, groupby="leiden", standard_scale="var")

Only one cluster (the one numbered 10 in the table above) shows these three genes: it is a **lymphoid (T/NK) cluster**, even if its top DE genes were uninformative.

> **Question** · Why are ribosomal genes (*RPS27, RPL28...*) so often among the top markers of a small cluster? Is it a biological result?
>
> <details><summary>Show answer</summary>
> Ribosomal protein genes are very abundant everywhere. Their relative proportion changes with the total RNA content and the quality of the cells: small cells (like lymphocytes) have a high relative fraction of ribosomal transcripts, because they express few other genes. In the test, the cluster stands out simply because the other clusters have a lower fraction. It is mostly a "compositional" effect, not a specific biological signature, which is why we need a targeted check with true lineage markers.
> </details>

### Safety check: translating the table to the numbering of *your* run

Cluster numbers are arbitrary labels. If you rerun the clustering on another computer or with other library versions, the *same* group of cells can receive a different number (for instance, two clusters can exchange their numbers), and a dictionary written with numbers would silently assign the wrong cell types.

To be safe, we do not trust the numbers: for each cluster of the table above we look for **the cluster of your run that shares its marker genes** (the one with the largest overlap between the genes of the table and its 30 top-ranked genes). This gives a translation `table number → your number`. If your numbering is identical to the one of the table, the translation is the identity.

In [ ]:
table_markers = {
    "0": ["KRT15", "DST", "COL17A1", "MIR205HG"], "16": ["S100A2", "COL17A1", "KRT5", "KRT14"],
    "11": ["SPINK5", "CALML5", "SLURP1", "SBSN"], "2": ["LGALS7", "LGALS7B", "KRT1", "PERP"],
    "6": ["RPLP0", "MT1X", "KRT5", "SNHG29"], "15": ["S100A14", "MT1G", "LY6D", "S100A9"],
    "3": ["HLA-DRB1", "HLA-DRA", "CD74", "HLA-DPB1"], "4": ["FTL", "CTSS", "AIF1", "CD68"],
    "10": ["RPS27", "B2M", "RPL28", "TMSB4X"], "12": ["TPSB2", "HPGD", "TPSAB1", "HPGDS"],
    "8": ["FTH1", "CFD", "SERPING1", "C1S"], "13": ["VIM", "LGALS1", "TPM2", "CAVIN3"],
    "14": ["SPARC", "COL3A1", "COL1A1", "LUM"], "7": ["MLANA", "DCT", "TYRP1"],
    "1": ["MCAM", "TAGLN", "CALD1", "MYL9"], "5": ["MMRN1", "CCL21", "TFF3"],
    "9": ["PECAM1", "SPARCL1", "GNG11"],
}
top30 = {
    cl: set(sc.get.rank_genes_groups_df(adata_sc, group=cl, key="rank_genes_leiden").head(30)["names"])
    for cl in adata_sc.obs["leiden"].cat.categories
}
# rows: clusters of the table; columns: clusters of our run; value: number of shared genes
overlap = pd.DataFrame({cl: {t: len(top30[cl] & set(genes)) for t, genes in table_markers.items()} for cl in top30})
remap = overlap.idxmax(axis=1).to_dict()          # table number -> number in our run
best = overlap.max(axis=1)

weak = [t for t in remap if best[t] < 2]
duplicated = len(set(remap.values())) != len(remap)
changed = {t: c for t, c in remap.items() if t != c}

if weak or duplicated:
    print("WARNING: the translation is ambiguous. Weak matches:", weak, "| duplicated:", duplicated)
    print("Re-derive the annotation by hand from the marker genes of YOUR clusters (dot plot and table above).")
elif changed:
    print("Your clusters are numbered differently from the table. Translation used (table -> yours):", changed)
else:
    print("OK: your cluster numbers are identical to the ones of the table.")

def to_our_numbering(d):
    """translate a dictionary written with the numbers of the table to the numbers of our run"""
    return {remap[k]: v for k, v in d.items()}

### Assign the cell types

We now encode our decisions in a Python **dictionary** (`cluster → cell type`, written with the numbers of the table) and apply it with `.map()`, after translating it to the numbering of our run.

In [ ]:
cluster_to_celltype = {
    "0": "Keratinocytes", "16": "Keratinocytes", "11": "Keratinocytes",
    "2": "Keratinocytes", "6": "Keratinocytes", "15": "Keratinocytes",
    "3": "Myeloid cells", "4": "Myeloid cells",
    "10": "Lymphoid cells",
    "12": "Mast cells",
    "8": "Fibroblasts", "13": "Fibroblasts", "14": "Fibroblasts",
    "7": "Melanocytes",
    "1": "Pericytes and smooth muscle cells",
    "5": "Endothelial cells", "9": "Endothelial cells",
}
adata_sc.obs["cell_type"] = adata_sc.obs["leiden"].map(to_our_numbering(cluster_to_celltype)).astype("category")

print(adata_sc.obs["cell_type"].value_counts().to_string())
sc.pl.embedding(adata_sc, basis="X_umap", color="cell_type", title="Cell type annotation")

> **Take-home** · Annotation is an interpretation: it combines data-driven markers, literature knowledge, and targeted checks. Several clusters can belong to the same cell type. Keep track of the evidence and treat the finer labels as hypotheses.

## Part 14 · Can we trust our annotation? Validation against the published one

> **Concept** · Our annotation is based on visual inspection of dot plots, a reasonable first pass but a subjective one. Because our cells are a subset of the same GEO dataset analyzed by the authors, we can run an objective, independent check: the authors assigned a cell type to every cell they kept (`GSE241132_cell_metadata.txt`, column `newMainCellTypes`), so we can match our cells to theirs **by barcode** and compare.
>
> A cautionary remark: this is a *consistency* check with a reference annotation, which is also the result of a (more elaborate) interpretation by the authors. It is useful, but it is not an absolute "ground truth".
>
> The barcode naming conventions differ: `{donor}{timepoint}_{barcode}` in the paper's table and `{barcode}-{timepoint}` in ours. First we reconcile the format.

In [ ]:
paper_meta = pd.read_csv(f"{SC_DATA}/GSE241132_cell_metadata.txt", sep="\t", index_col=0)

print("Paper metadata barcode format (first 3):", paper_meta.index[:3].tolist())
print("Our barcode format (first 3):          ", adata_sc.obs_names[:3].tolist())
print("\nPaper metadata columns:", paper_meta.columns.tolist())

In [ ]:
DONOR = "PWH27"

def to_paper_barcode(adata_barcode, donor=DONOR):
    core, tp = adata_barcode.rsplit("-", 1)      # "AAACCCAAGTATCCTG-1", "D0"
    return f"{donor}{tp}_{core}"

adata_sc.obs["paper_barcode"] = [to_paper_barcode(bc) for bc in adata_sc.obs_names]

paper_meta_pwh27 = paper_meta[paper_meta.index.str.startswith(DONOR)]
adata_sc.obs["paper_newMainCellTypes"] = adata_sc.obs["paper_barcode"].map(paper_meta_pwh27["newMainCellTypes"])

n_matched = adata_sc.obs["paper_newMainCellTypes"].notna().sum()
print(f"Matched {n_matched} / {adata_sc.n_obs} cells to the published annotation ({100 * n_matched / adata_sc.n_obs:.1f}%)")

> **Question** · Some of our cells do not have a match in the paper's table. Why could that be?
>
> <details><summary>Show answer</summary>
> The authors applied their own, slightly different filters (QC, doublet detection with two methods, removal of low-quality clusters). Cells that they discarded are absent from their table, and cells that we discarded are absent from ours. Different filters, different cells. It is a good reminder that preprocessing choices change the dataset itself.
> </details>

### Cross-tabulation: our clusters vs. the published cell types

For each of our clusters, how did the authors classify its cells?

In [ ]:
pd.crosstab(adata_sc.obs["leiden"], adata_sc.obs["paper_newMainCellTypes"])

Each row should be dominated by a single column. Let's measure the agreement: we translate the paper's labels into our vocabulary (we count the *Hair follicle* cells of the paper as keratinocytes, since the hair follicle is an epithelial structure made of keratinocyte-lineage cells) and compute, per cluster, the fraction of cells whose published label matches ours.

In [ ]:
paper_to_ours = {
    "Keratinocyte": "Keratinocytes", "Hair follicle": "Keratinocytes",
    "Fibroblast": "Fibroblasts", "Myeloid": "Myeloid cells", "Lymphoid": "Lymphoid cells",
    "Endothelial": "Endothelial cells", "Mast": "Mast cells", "Melanocyte": "Melanocytes",
    "Pericyte & Smooth muscle": "Pericytes and smooth muscle cells",
}
obs = adata_sc.obs
obs["paper_in_our_vocabulary"] = obs["paper_newMainCellTypes"].map(paper_to_ours)

comparable = obs[obs["paper_in_our_vocabulary"].notna()]             # cells with a label that exists in our vocabulary
agree = (comparable["cell_type"].astype(str) == comparable["paper_in_our_vocabulary"])
print(f"Overall agreement: {100 * agree.mean():.1f}% of {len(comparable)} comparable cells")

per_cluster = agree.groupby(comparable["leiden"], observed=True).mean().mul(100).round(1)
print("\nAgreement per cluster (%):")
print(per_cluster.to_string())

The agreement is high in all clusters. The disagreements are informative as well: they are mostly small populations that our resolution did not separate (for example, a handful of Schwann cells absorbed in a fibroblast cluster, or some lymphoid cells inside a myeloid cluster). This is the **resolution limit** of our clustering, not a mistake in the annotation of the main groups.

**Our decision.** We keep our own marker-based annotation as the final one, and we use the published labels as a validation (not as a replacement).

> **Take-home** · An annotation should be validated by independent evidence. Here, a high agreement with the published labels gives us confidence in the **major cell types**. Note that this validation does *not* cover the finer state labels, which we treat as hypotheses.

In [ ]:
# Save checkpoint: cell type annotation and mapping to the paper
adata_sc.write_h5ad(f"{OUT}/adata_sc_leiden_res0.2_annotated.h5ad")

# To reload it later:
# adata_sc = sc.read_h5ad(f"{OUT}/adata_sc_leiden_res0.2_annotated.h5ad")

## Part 15 · Cell states and tissue composition during healing

### A finer annotation: cell states

> **Concept** · Cell types are the "big categories"; **cell states** are the different functional or activation conditions *within* a type (a basal vs. a differentiated keratinocyte, a quiescent vs. an activated fibroblast). Our 17 clusters are at the level of states. We assign one descriptive label to every cluster. These labels follow from the markers (Part 13) and are **hypotheses**, in particular for the stress/ribosomal cluster 6, whose markers are not specific.

In [ ]:
cluster_to_state = {
    "0":  "Keratinocytes_basal_A",
    "16": "Keratinocytes_basal_B",
    "11": "Keratinocytes_differentiated",
    "2":  "Keratinocytes_suprabasal",
    "6":  "Keratinocytes_ribosomal_stress",
    "15": "Keratinocytes_wound_activated",
    "3":  "Myeloid_MHCII_high",
    "4":  "Macrophages",
    "10": "Lymphoid_T_NK",
    "12": "Mast_cells",
    "8":  "Fibroblasts_CFD",
    "13": "Fibroblasts_myofibroblast_like",
    "14": "Fibroblasts_ECM_high",
    "7":  "Melanocytes",
    "1":  "Pericytes_SMC",
    "5":  "Endothelial_lymphatic",
    "9":  "Endothelial_vascular",
}
adata_sc.obs["cell_state"] = adata_sc.obs["leiden"].map(to_our_numbering(cluster_to_state)).astype("category")
print(adata_sc.obs["cell_state"].value_counts().to_string())

adata_sc.write_h5ad(f"{OUT}/adata_sc_annotated_cell_state.h5ad")
# To reload it later:
# adata_sc = sc.read_h5ad(f"{OUT}/adata_sc_annotated_cell_state.h5ad")

### How does the tissue composition change over time?

We have annotated the cells; we can now ask the biological question of the study: **which cell types are present at each timepoint?** The simplest answer is the *proportion* of every cell type in every sample.

In [ ]:
prop_type = pd.crosstab(adata_sc.obs["sample"], adata_sc.obs["cell_type"], normalize="index").loc[SC_SAMPLES] * 100
ax = prop_type.plot.bar(stacked=True, figsize=(9, 4.5), colormap="tab20", width=0.8)
ax.set_ylabel("% of cells in the sample"); ax.set_xlabel("")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False)
ax.set_title("Cell type composition of each sample")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

prop_type.round(1)

In [ ]:
prop_state = pd.crosstab(adata_sc.obs["sample"], adata_sc.obs["cell_state"], normalize="index").loc[SC_SAMPLES] * 100
fig, ax = plt.subplots(figsize=(11, 4.5))
sns.heatmap(prop_state.T, annot=True, fmt=".1f", cmap="Blues", cbar_kws={"label": "% of the sample"}, ax=ax)
ax.set_xlabel(""); ax.set_ylabel(""); ax.set_title("Cell state composition of each sample (%)")
plt.tight_layout(); plt.show()

> **Question** · Which cell types or states increase or decrease during healing? Do they make biological sense (think of what happens in a wound: inflammation, migration, remodeling)?
>
> <details><summary>Show answer</summary>
> You should see, for example, changes in the proportion of immune cells (myeloid cells, lymphoid cells) in the wound samples compared to intact skin, and changes in the keratinocyte and fibroblast states. These are consistent with the inflammatory phase (early immune infiltration), re-epithelialization (keratinocytes in wound-activated state) and tissue remodeling (fibroblasts) of wound healing. Read your own table: which states are almost absent in D0 and appear later?
> </details>

> **Caution · compositional data.** These are **proportions**, which add up to 100% in each sample: if one population grows a lot, all the others decrease in percentage even if their number did not change. Moreover: (i) we have **one donor and one sample per timepoint**, so we cannot say how much of the difference is due to biology and how much to the biopsy / dissociation (e.g. dissociation does not recover all the cell types with the same efficiency); (ii) there is no statistical test here. The figure is a description, not a proof. To make solid statements about changes in abundance we need biological replicates and dedicated statistical methods.

## Part 16 · Take-home messages

What did we learn about scRNA-seq data analysis?

1. **Know your data.** A count matrix is a very sparse sample of molecules from each cell. A zero is not always a "not expressed".
2. **QC is a decision, not a formula.** Look at the distributions, per sample, before choosing a threshold; report it.
3. **Normalization removes technical differences in depth** so that cells can be compared; the log compresses the dynamic range. Keep the raw counts.
4. **Select informative genes** (HVGs): less noise, less computation, same biology.
5. **PCA** summarizes the data in a few "gene programs". The exact number of PCs matters less than people think, provided it is not too low.
6. **The UMAP is a picture, not a measurement.** It changes with the seed and the parameters; do not read distances or sizes from it.
7. **Do not correct batch effects blindly.** Ask what your batch variable represents; if it is confounded with the biology, correction can erase the signal.
8. **There is no true number of clusters.** The resolution is a choice of detail; statistical criteria help, biology decides.
9. **Markers connect clusters to cell types**; the p-values of a DE test on clusters defined with the same data are only a ranking tool.
10. **Annotation is interpretation**: keep the evidence, validate independently, treat finer labels as hypotheses.
11. **Reproducibility.** Fix seeds, record versions, save checkpoints, keep raw counts, name your objects explicitly (do not let functions overwrite previous results).

### What we did not do (and where to go next)

Natural continuations of this analysis, each of which could be a practical session:

- **Sub-clustering** of one population (for example, keratinocytes): repeat the whole pipeline on a subset to resolve finer states.
- **Trajectory / pseudotime analysis** (PAGA, diffusion pseudotime): order cells along a process, such as keratinocyte differentiation or migration.
- **Pathway / gene set enrichment** (GO, Reactome): from a list of genes to biological processes.
- **Differential abundance** (e.g. Milo) and **pseudobulk differential expression** between timepoints, which require biological replicates.
- **Batch correction by donor** with a multi-donor design.
- **Cell–cell communication** (ligand–receptor analysis).
- **Integration with spatial transcriptomics** of the same tissue.

> **Final question** · If you had to explain to a colleague, in three sentences, how we went from a table of molecule counts to a map of the cell types of the wound, what would you say?